# Train / val / test split

Stratified podela Pandora18K (70/15/15), seed=42. Rezultat se cuva u `data/splits/`.

In [2]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

ROOT = Path("..")
DATA_DIR = ROOT / "Pandora_18k"
OUT_DIR = ROOT / "data" / "splits"

SEED = 42
VAL_SIZE = 0.15
TEST_SIZE = 0.15
EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}

In [3]:
rows = []
class_dirs = sorted(p for p in DATA_DIR.iterdir() if p.is_dir())

for class_dir in class_dirs:
    label = class_dir.name
    for path in class_dir.rglob("*"):
        if path.is_file() and path.suffix.lower() in EXTS:
            # relativna putanja od root-a projekta
            rel = path.relative_to(ROOT).as_posix()
            rows.append({"filepath": rel, "label": label})

df = pd.DataFrame(rows)
print(f"ukupno: {len(df)}, klasa: {df['label'].nunique()}")
df.head()

ukupno: 18038, klasa: 18


,filepath,label
0,Pandora_18k/01_Byzantin_Iconography/Cimabue/ma...,01_Byzantin_Iconography
1,Pandora_18k/01_Byzantin_Iconography/Cimabue/vi...,01_Byzantin_Iconography
2,Pandora_18k/01_Byzantin_Iconography/Cimabue/th...,01_Byzantin_Iconography
3,Pandora_18k/01_Byzantin_Iconography/Cimabue/kr...,01_Byzantin_Iconography
4,Pandora_18k/01_Byzantin_Iconography/Cimabue/ma...,01_Byzantin_Iconography


In [4]:
train_df, temp_df = train_test_split(
    df,
    test_size=VAL_SIZE + TEST_SIZE,
    stratify=df["label"],
    random_state=SEED,
)

relative_test = TEST_SIZE / (VAL_SIZE + TEST_SIZE)
val_df, test_df = train_test_split(
    temp_df,
    test_size=relative_test,
    stratify=temp_df["label"],
    random_state=SEED,
)

print("train:", len(train_df))
print("val:  ", len(val_df))
print("test: ", len(test_df))

train: 12626
val:   2706
test:  2706


In [6]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

train_df.sort_values(["label", "filepath"]).to_csv(OUT_DIR / "train.csv", index=False)
val_df.sort_values(["label", "filepath"]).to_csv(OUT_DIR / "val.csv", index=False)
test_df.sort_values(["label", "filepath"]).to_csv(OUT_DIR / "test.csv", index=False)

print("sacuvano u", OUT_DIR.resolve())

# provera da li ima preklapanja
s_train, s_val, s_test = set(train_df.filepath), set(val_df.filepath), set(test_df.filepath)
print("overlap train-val:", len(s_train & s_val))
print("overlap train-test:", len(s_train & s_test))
print("overlap val-test:", len(s_val & s_test))

train_df["label"].value_counts().sort_index()

sacuvano u /home/anja/Desktop/ML/renaissance-to-pop-art-study/data/splits
overlap train-val: 0
overlap train-test: 0
overlap val-test: 0


label
01_Byzantin_Iconography    593
02_Early_Renaissance       526
03_Northern_Renaissance    575
04_High_Renaissance        582
05_Baroque                 693
06_Rococo                  582
07_Romanticism             627
08_Realism                 839
09_Impressionism           880
10_Post_Impressionism      893
11_Expressionism           719
12_Symbolism               740
13_Fauvism                 503
14_Cubism                  859
15_Surrealism              750
16_AbstractArt             744
17_NaiveArt                737
18_PopArt                  784
Name: count, dtype: int64

## Provera loadera (`dataset.py`)

In [7]:
import sys
sys.path.append(str((ROOT / "src").resolve()))

from dataset import make_datasets

train_ds, val_ds, test_ds, names = make_datasets()
print("klasa:", len(names))
print("batcheva train/val/test:", len(train_ds), len(val_ds), len(test_ds))

x, y = next(iter(train_ds))
print("batch x:", x.shape, "y:", y.shape)

klasa: 18
batcheva train/val/test: 395 85 85
batch x: (32, 224, 224, 3) y: (32, 18)
